<hr>

# 🧫 DATA CLEANING 


<style>
h1 {
    text-align: left;
    color: blue;
    font-weight: bold;
}

</style>
<hr>

<style>
h3 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

### 📂 IMPORTs

In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sns

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200) # to display all columns in the dataframe

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DISPLAY_DATA FUNCTION 

In [ ]:
import pandas as pd
# define function display_data to explore each dataset
def display_data(df: pd.DataFrame):
    # ----------------------------------------
    # Display shape and head of the dataframe
    # ----------------------------------------
    print(f"{df.name} Dataset Shape: {df.shape[0]} rows and {df.shape[1]} columns\n")
    display(df.head(3))
    print(100*"-" + "\n")

    # ----------------------------------------------------------
    # Display data types & missing values of each column in df
    # ----------------------------------------------------------
    print("Data Types & Missing Values of Each Column:")
    display(df.info())

    # ---------------------------------------------------
    # Display Info & unique values for each column in df    
    # ---------------------------------------------------
    for col in df.columns:
        unique_vals = df[col].unique()
        print(f"Column: {col}")
        print(f"Unique values ({len(unique_vals)}): {unique_vals}\n")

    print(f"\nData types check:")
    for col in df.columns:
        dtype = df[col].dtype
        unique_vals = df[col].nunique()
        print(f"  ➡️{col:<25} {str(dtype):<10} [{unique_vals}] unique values")

<hr>

## 0 - DATA CLEANING OVERVIEW


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

Note observations and changes to be made on the data:
- note a
- note b
- note c
- change 1
- change 2
- change 3

<hr>

## 1 - STANDARDIZE NAMES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### ➕ COMBINE - 6 TXT FILES INTO 1 CSV
Dropped unnecessary columns

In [2]:
import pandas as pd

# List of DVF files
dvf_files = [
    "../data/raw/ValeursFoncieres-2020-S2.txt",
    "../data/raw/ValeursFoncieres-2021.txt",
    "../data/raw/ValeursFoncieres-2022.txt",
    "../data/raw/ValeursFoncieres-2023.txt",
    "../data/raw/ValeursFoncieres-2024.txt",
    "../data/raw/ValeursFoncieres-2025-S1.txt"
]

# Columns to keep
usecols = [
    "Date mutation",
    "Nature mutation",
    "Valeur fonciere",
    "Commune",
    "Type local",
    "Surface reelle bati",
    "Nombre pieces principales",
    "Nombre de lots",
    "Surface terrain", 
    "No voie",    # street number
    "B/T/Q",      # building / type / quartier
    "Voie",       # street name
    "Code postal" # postal code
]

# Output CSV
output_file = "../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv"

# Fix mixed types
dtype_dict = {
    "Code postal": str,
    "Voie": str,
    "No voie": str,
    "B/T/Q": str,
    "Type local": str,
    "department_code": str,
    "surface_type": str

}

with open(output_file, "w", newline="", encoding="utf-8") as f_out:
    header_written = False

    for file in dvf_files:
        print(f"Processing: {file}")

        for chunk in pd.read_csv(
            file,
            sep="|",
            usecols=usecols,
            chunksize=50_000,
            encoding="latin-1",
            decimal=",",
            low_memory=False,
            dtype=dtype_dict
        ):
            # -------------------------
            # FIX ENCODING
            # -------------------------
            chunk["Type local"] = (
                chunk["Type local"]
                .astype(str)
                .str.encode("latin-1", errors="ignore")
                .str.decode("utf-8", errors="ignore")
                .str.strip()
            )

            # -------------------------
            # CONVERT NUMERIC
            # -------------------------
            chunk["Valeur fonciere"] = pd.to_numeric(chunk["Valeur fonciere"], errors="coerce")
            chunk["Surface reelle bati"] = pd.to_numeric(chunk["Surface reelle bati"], errors="coerce")
            chunk["Surface terrain"] = pd.to_numeric(chunk["Surface terrain"], errors="coerce")

            # Drop rows with missing price
            chunk = chunk.dropna(subset=["Valeur fonciere"])

            # -------------------------
            # PRICE PER M² (SMART FALLBACK)
            # -------------------------
            chunk["effective_surface"] = chunk["Surface reelle bati"]

            # If built surface is missing or zero → use terrain
            mask = chunk["effective_surface"].isna() | (chunk["effective_surface"] == 0)
            chunk.loc[mask, "effective_surface"] = chunk.loc[mask, "Surface terrain"]

            # Compute price per m²
            chunk["value_per_m2"] = chunk["Valeur fonciere"] / chunk["effective_surface"]

            # Clean invalid values
            chunk.loc[
                chunk["effective_surface"].isna() | (chunk["effective_surface"] == 0),
                "value_per_m2"
            ] = None

            # Optional: track which surface was used
            chunk["surface_type"] = "bati"
            chunk.loc[mask, "surface_type"] = "terrain"

            # -------------------------
            # EXTRACT DEPARTEMENT
            # -------------------------
            chunk["department_code"] = chunk["Code postal"].astype(str).str[:2]

            # -------------------------
            # WRITE TO CSV
            # -------------------------
            chunk.to_csv(
                f_out,
                index=False,
                header=not header_written
            )

            header_written = True
            del chunk  # free memory

print(f"✅ TXT Files aggregated & converted into one CSV File")
print(f"✅ CSV File saved at {output_file}")

Processing: ../data/raw/ValeursFoncieres-2020-S2.txt
Processing: ../data/raw/ValeursFoncieres-2021.txt
Processing: ../data/raw/ValeursFoncieres-2022.txt
Processing: ../data/raw/ValeursFoncieres-2023.txt
Processing: ../data/raw/ValeursFoncieres-2024.txt
Processing: ../data/raw/ValeursFoncieres-2025-S1.txt
✅ TXT Files aggregated & converted into one CSV File
✅ CSV File saved at ../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv


### ⭐ **Valeurs Foncieres 2020-2025**
Standardize column names

| Original Column Name      |      Standardized      | Notes / Description                                           |
| ------------------------- | ---------------------- | ------------------------------------------------------------ |
| **Date mutation**             | `transaction_date`       | Date when the property was sold                        |
| **Nature mutation**           | `transaction_type`       | Nature of transaction (sale, donation, etc.)                 |
| **Valeur fonciere**           | `property_value`         | Transaction price in euros                                   |
| **No voie**                   | `street_number`          | Street number of the property                                |
| **B/T/Q**                     | `btq_code`               | Bâtiment / Type / Quartier code for identification           |
| **Voie**                      | `street_name`            | Full street name                                             |
| **Code postal**               | `postal_code`            | Postal code (categorical, not numeric)                       |
| **Commune**                   | `commune_name`                | Commune name                                                 |
| **Nombre de lots**            | `number_of_lots`         | Number of lots in the property                               |
| **Type local**                | `property_type`          | Property type (Apartment, House, etc.)                       |
| **Surface reelle bati**       | `building_real_surface`  | Built area in square meters                                  |
| **Nombre pieces principales** | `main_rooms_count`        | Number of main rooms (living / bedrooms)                     |
| **Surface terrain**           | `land_surface`           | Land area in square meters                                   |
| **effective_surface**              | `effective_surface`      | Surface used to calculate price/m² (building or land)        |
| **value_per_m2**                   | `value_per_m2`           | Value per m² in euros                                        |
| **surface_type**              | `surface_type`           | Indicates if effective_surface comes from a building or land |
| **departement**               | `department_code`        | 2-digit INSEE department code                                |


<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DISPLAY HEAD - RAW Valeurs Foncieres

In [3]:
import pandas as pd
df = pd.read_csv("../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv")

# Display df DataFrame
print("RAW Valeur Foncieres:")
display(df.head())
print("Dataset Shape:", df.shape[0], "rows and", df.shape[1], "columns\n")

C:\Users\sboub\AppData\Local\Temp\ipykernel_8064\3339224207.py:2: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv")


RAW Valeur Foncieres:


,Date mutation,Nature mutation,Valeur fonciere,No voie,B/T/Q,Voie,Code postal,Commune,Nombre de lots,Type local,Surface reelle bati,Nombre pieces principales,Surface terrain,effective_surface,value_per_m2,surface_type,department_code
0,01/07/2020,Vente,31234.16,NaN,NaN,SAINT JULIEN,1560.0,SAINT-JULIEN-SUR-REYSSOUZE,0,NaN,NaN,NaN,1192.0,1192.0,26.203154,terrain,15
1,01/07/2020,Vente,278000.00,NaN,NaN,A LA PEROUSE,1250.0,CORVEISSIAT,0,NaN,NaN,NaN,10092.0,10092.0,27.546572,terrain,12
2,01/07/2020,Vente,278000.00,NaN,NaN,A LA PEROUSE,1250.0,CORVEISSIAT,0,NaN,NaN,NaN,4570.0,4570.0,60.831510,terrain,12
3,01/07/2020,Vente,278000.00,NaN,NaN,AUX COMMUNS,1250.0,CORVEISSIAT,0,NaN,NaN,NaN,5750.0,5750.0,48.347826,terrain,12
4,01/07/2020,Vente,278000.00,NaN,NaN,EN COMBARNAUD,1250.0,SIMANDRE-SUR-SURAN,0,NaN,NaN,NaN,648170.0,648170.0,0.428900,terrain,12


Dataset Shape: 19908349 rows and 17 columns



<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DISPLAY UNIQUE VALUES - RAW Valeurs Foncieres

In [4]:
# Display data types & missing values of each column in df
print("Data Types & Missing Values of Each Column:")
display(df.info())

# Loop through each column and print unique values
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"Column: {col}")
    print(f"Unique values ({len(unique_vals)}): {unique_vals}\n")

print(f"\nData types check:")
for col in df.columns:
    dtype = df[col].dtype
    unique_vals = df[col].nunique()
    print(f"  ➡️{col:<25} {str(dtype):<10} [{unique_vals}] unique values")

Data Types & Missing Values of Each Column:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19908349 entries, 0 to 19908348
Data columns (total 17 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   Date mutation              object 
 1   Nature mutation            object 
 2   Valeur fonciere            float64
 3   No voie                    float64
 4   B/T/Q                      object 
 5   Voie                       object 
 6   Code postal                float64
 7   Commune                    object 
 8   Nombre de lots             int64  
 9   Type local                 object 
 10  Surface reelle bati        float64
 11  Nombre pieces principales  float64
 12  Surface terrain            float64
 13  effective_surface          float64
 14  value_per_m2               float64
 15  surface_type               object 
 16  department_code            object 
dtypes: float64(8), int64(1), object(8)
memory usage: 2.5+ GB


None

Column: Date mutation
Unique values (1804): ['01/07/2020' '04/07/2020' '02/07/2020' ... '01/05/2025' '18/05/2025'
 '05/01/2025']

Column: Nature mutation
Unique values (6): ['Vente' "Vente en l'Ã©tat futur d'achÃ¨vement"
 'Vente terrain Ã\xa0 bÃ¢tir' 'Echange' 'Adjudication' 'Expropriation']

Column: Valeur fonciere
Unique values (466667): [3.123416e+04 2.780000e+05 3.985000e+03 ... 7.088090e+05 5.503861e+05
 2.441758e+07]

Column: No voie
Unique values (9121): [  nan  347. 1084. ... 8562. 8301. 8423.]

Column: B/T/Q
Unique values (41): [nan 'B' 'C' 'A' 'F' 'L' 'T' 'X' 'O' 'Z' 'D' 'G' 'Q' 'Y' 'I' 'E' 'N' 'H'
 '1' 'W' 'V' 'U' 'S' 'R' 'P' 'M' 'K' 'J' '0' '5' '9' '8' 'b' '7' '2' '4'
 '3' '6' '-' '.' '*']

Column: Voie
Unique values (1125220): ['SAINT JULIEN' 'A LA PEROUSE' 'AUX COMMUNS' ... 'SAINT PIERRE AMELOT'
 'SAINT HONORE D EYLAU' 'DES FOSSES SAINT MARCEL']

Column: Code postal
Unique values (5872): [ 1560.  1250.  1310. ... 97380. 97314. 97330.]

Column: Commune
Unique values (31569

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### ✏️ RENAME - COLUMNS

In [5]:
import pandas as pd

# -------------------
# Load main dataset
# -------------------
df = pd.read_csv("../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv")

# ---------------------------------------------
# Define the Renaming Dictionary
# ---------------------------------------------
rename_columns = {
    "Date mutation": "transaction_date",
    "Nature mutation": "transaction_type",
    "Valeur fonciere": "property_value",
    "No voie": "street_number",
    "B/T/Q": "btq_code",
    "Voie": "street_name",
    "Code postal": "postal_code",
    "Commune": "commune_name",
    "Nombre de lots": "number_of_lots",
    "Type local": "property_type",
    "Surface reelle bati": "building_real_surface",
    "Nombre pieces principales": "main_rooms_count",
    "Surface terrain": "land_surface",
    "surface_used": "effective_surface",
    "Prix_m2": "value_per_m2",
    "surface_type": "surface_type",
    "departement": "department_code"
}

# ---------------------------------------------
# Rename columns using the rename Dictionary
# ---------------------------------------------
df.rename(columns=rename_columns, inplace=True)

# --------------------------------
# Save dataset in a new CSV file
# --------------------------------
df.to_csv("../data/processed/INTERIM_01_standard_ValeursFoncieres.csv", index=False)

print(f"✅ Dataset standardized and saved at ../data/processed/INTERIM_01_standard_ValeursFoncieres.csv")

C:\Users\sboub\AppData\Local\Temp\ipykernel_8064\2975596484.py:6: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv")


✅ Dataset standardized and saved at ../data/processed/INTERIM_01_standard_ValeursFoncieres.csv


In [ ]:
### ⭐ **Valeurs Foncieres 2020-2025**
Standardize column names

| Original Column Name      |      Standardized      | Notes / Description                                           |
| ------------------------- | ---------------------- | ------------------------------------------------------------ |
| Date mutation             | transaction_date       | Date when the property was sold                              |
| Nature mutation           | transaction_type       | Nature of transaction (sale, donation, etc.)                 |
| Valeur fonciere           | property_value         | Transaction price in euros                                   |
| No voie                   | street_number          | Street number of the property                                |
| B/T/Q                     | btq_code               | Bâtiment / Type / Quartier code for identification           |
| Voie                      | street_name            | Full street name                                             |
| Code postal               | postal_code            | Postal code (categorical, not numeric)                       |
| Commune                   | commune_name           | Commune name                                                 |
| Nombre de lots            | number_of_lots         | Number of lots in the property                               |
| Type local                | property_type          | Property type (Apartment, House, etc.)                       |
| Surface reelle bati       | building_real_surface  | Built area in square meters                                  |
| Nombre pieces principales | main_rooms_count       | Number of main rooms (living / bedrooms)                     |
| Surface terrain           | land_surface           | Land area in square meters                                   |
| surface_used              | effective_surface      | Surface used to calculate price/m² (building or land)        |
| Prix_m2                   | value_per_m2           | Value per m² in euros                                        |
| surface_type              | surface_type           | Indicates if effective_surface comes from a building or land |
| departement               | department_code        | 2-digit INSEE department code                                |


<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🟨 DISPLAY - Standard ValeursFoncieres

<hr>

## 2 - INVALID VALUES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### LOAD & DISPLAY STANDARDIZED - Valeurs Foncieres

In [ ]:
import pandas as pd
df = pd.read_csv("../data/processed/INTERIM_01_standard_ValeursFoncieres.csv")
print("Standardized Valeurs Foncieres Dataset Shape:", df.shape[0], "rows and", df.shape[1], "columns")
display(df.head())

In [ ]:
# Display Standardized DataFrame
print("Standardized DataFrame:")
print(df.shape[0], "rows and", df.shape[1], "columns")
display(df.head())

# Display standardized column names
print("Standardized Column Names:")
display(df.info())

Standardized DataFrame:
19908349 rows and 17 columns


,transaction_date,transaction_type,property_value,street_number,btq_code,street_name,postal_code,commune_name,number_of_lots,property_type,building_real_surface,main_rooms_count,land_surface,effective_surface,value_per_m2,surface_type,department_code
0,01/07/2020,Vente,31234.16,NaN,NaN,SAINT JULIEN,1560.0,SAINT-JULIEN-SUR-REYSSOUZE,0,NaN,NaN,NaN,1192.0,1192.0,26.203154,terrain,15
1,01/07/2020,Vente,278000.00,NaN,NaN,A LA PEROUSE,1250.0,CORVEISSIAT,0,NaN,NaN,NaN,10092.0,10092.0,27.546572,terrain,12
2,01/07/2020,Vente,278000.00,NaN,NaN,A LA PEROUSE,1250.0,CORVEISSIAT,0,NaN,NaN,NaN,4570.0,4570.0,60.831510,terrain,12
3,01/07/2020,Vente,278000.00,NaN,NaN,AUX COMMUNS,1250.0,CORVEISSIAT,0,NaN,NaN,NaN,5750.0,5750.0,48.347826,terrain,12
4,01/07/2020,Vente,278000.00,NaN,NaN,EN COMBARNAUD,1250.0,SIMANDRE-SUR-SURAN,0,NaN,NaN,NaN,648170.0,648170.0,0.428900,terrain,12


Standardized Column Names:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19908349 entries, 0 to 19908348
Data columns (total 17 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   transaction_date       object 
 1   transaction_type       object 
 2   property_value         float64
 3   street_number          float64
 4   btq_code               object 
 5   street_name            object 
 6   postal_code            float64
 7   commune_name           object 
 8   number_of_lots         int64  
 9   property_type          object 
 10  building_real_surface  float64
 11  main_rooms_count       float64
 12  land_surface           float64
 13  effective_surface      float64
 14  value_per_m2           float64
 15  surface_type           object 
 16  department_code        object 
dtypes: float64(8), int64(1), object(8)
memory usage: 2.5+ GB


None

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DISPLAY UNIQUE VALUES - Valeurs Foncieres

In [ ]:
# Loop through each column and print unique values
print(100 * "-")
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"➡️ {col} ({len(unique_vals)}) 🧾 unique values:")
    print(f"{unique_vals}")
    print(100 * "-")

----------------------------------------------------------------------------------------------------
➡️ transaction_date (1804) 🧾 unique values:
['01/07/2020' '04/07/2020' '02/07/2020' ... '01/05/2025' '18/05/2025'
 '05/01/2025']
----------------------------------------------------------------------------------------------------
➡️ transaction_type (6) 🧾 unique values:
['Vente' "Vente en l'Ã©tat futur d'achÃ¨vement"
 'Vente terrain Ã\xa0 bÃ¢tir' 'Echange' 'Adjudication' 'Expropriation']
----------------------------------------------------------------------------------------------------
➡️ property_value (466667) 🧾 unique values:
[3.123416e+04 2.780000e+05 3.985000e+03 ... 7.088090e+05 5.503861e+05
 2.441758e+07]
----------------------------------------------------------------------------------------------------
➡️ street_number (9121) 🧾 unique values:
[  nan  347. 1084. ... 8562. 8301. 8423.]
----------------------------------------------------------------------------------------------

#### COMMUNES CSV (for consistency in commune_name values)

In [ ]:
df_communes = pd.read_csv("../data/raw/communes-france-2025.csv")
df_communes = df_communes.drop(columns=["Unnamed: 0"])
display(df_communes.head())
df_communes.info()

# Loop through each column and print unique values
print(100 * "-")
for col in df_communes.columns:
    unique_vals = df_communes[col].unique()
    print(f"➡️ {col} ({len(unique_vals)}) 🧾 unique values:")
    print(f"{unique_vals}")
    print(100 * "-")


# keep these columns
# code_insee
# nom_standard
# nom_sans_pronom
# nom_a
# nom_de
# nom_sans_accent
# nom_standard_majuscule
code_insee	nom_standard	nom_sans_pronom	nom_a	nom_de	nom_sans_accent	nom_standard_majuscule
reg_code
dep_code
dep_nom
code_postal

df_communes_clean = df_communes[["Nom officiel de la commune", "Code INSEE"]].copy()

C:\Users\sboub\AppData\Local\Temp\ipykernel_8064\3610903181.py:1: DtypeWarning: Columns (1,12,14,16,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df_communes = pd.read_csv("../data/raw/communes-france-2025.csv")


,code_insee,nom_standard,nom_sans_pronom,nom_a,nom_de,nom_sans_accent,nom_standard_majuscule,typecom,typecom_texte,reg_code,reg_nom,dep_code,dep_nom,canton_code,canton_nom,epci_code,epci_nom,academie_code,academie_nom,code_postal,codes_postaux,zone_emploi,code_insee_centre_zone_emploi,code_unite_urbaine,nom_unite_urbaine,taille_unite_urbaine,type_commune_unite_urbaine,statut_commune_unite_urbaine,population,superficie_hectare,superficie_km2,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre,grille_densite,grille_densite_texte,niveau_equipements_services,niveau_equipements_services_texte,gentile,url_wikipedia,url_villedereve
0,01001,L'Abergement-Clémenciat,Abergement-Clémenciat,à Abergement-Clémenciat,de l'Abergement-Clémenciat,l-abergement-clemenciat,L'ABERGEMENT-CLÉMENCIAT,COM,commune,84,Auvergne-Rhône-Alpes,01,Ain,0108,Châtillon-sur-Chalaronne,200069193,CC de la Dombes,10,Lyon,1400.0,01400,8405.0,01053,01000,NaN,0.0,HORS UNITE URBAINE,H,832,1565,16,53.0,242,206.0,272.0,46.151,4.921,46.153,4.926,6,Rural à habitat dispersé,0.0,communes non pôle,NaN,https://fr.wikipedia.org/wiki/fr:L'Abergement-...,https://villedereve.fr/ville/01001-l-abergemen...
1,01002,L'Abergement-de-Varey,Abergement-de-Varey,à Abergement-de-Varey,de l'Abergement-de-Varey,l-abergement-de-varey,L'ABERGEMENT-DE-VAREY,COM,commune,84,Auvergne-Rhône-Alpes,01,Ain,0101,Ambérieu-en-Bugey,240100883,CC de la Plaine de l'Ain,10,Lyon,1640.0,01640,8405.0,01053,01000,NaN,0.0,HORS UNITE URBAINE,H,267,912,9,29.0,483,290.0,748.0,46.007,5.423,46.009,5.428,6,Rural à habitat dispersé,0.0,communes non pôle,"Abergementais, Abergementaises",https://fr.wikipedia.org/wiki/fr:L'Abergement-...,https://villedereve.fr/ville/01002-l-abergemen...
2,01004,Ambérieu-en-Bugey,Ambérieu-en-Bugey,à Ambérieu-en-Bugey,d'Ambérieu-en-Bugey,amberieu-en-bugey,AMBÉRIEU-EN-BUGEY,COM,commune,84,Auvergne-Rhône-Alpes,01,Ain,0101,Ambérieu-en-Bugey,240100883,CC de la Plaine de l'Ain,10,Lyon,1500.0,"01500, 01501, 01504, 01503, 01502, 01505, 01506",8405.0,01053,01303,Ambérieu-en-Bugey,3.0,UNITE URBAINE,C,14854,2448,24,607.0,379,237.0,753.0,45.958,5.360,45.961,5.373,2,Centres urbains intermédiaires,3.0,centres structurants d'équipements et de services,"Ambarrois, Ambarroises",https://fr.wikipedia.org/wiki/fr:Ambérieu-en-B...,https://villedereve.fr/ville/01004-amberieu-en...
3,01005,Ambérieux-en-Dombes,Ambérieux-en-Dombes,à Ambérieux-en-Dombes,d'Ambérieux-en-Dombes,amberieux-en-dombes,AMBÉRIEUX-EN-DOMBES,COM,commune,84,Auvergne-Rhône-Alpes,01,Ain,0122,Villars-les-Dombes,200042497,CC Dombes Saône Vallée,10,Lyon,1330.0,01330,8434.0,69264,01000,NaN,0.0,HORS UNITE URBAINE,H,1897,1605,16,118.0,290,265.0,302.0,45.996,4.903,45.996,4.912,5,Bourgs ruraux,1.0,centres locaux d'équipements et de services,Ambarrois,https://fr.wikipedia.org/wiki/fr:Ambérieux-en-...,https://villedereve.fr/ville/01005-amberieux-e...
4,01006,Ambléon,Ambléon,à Ambléon,d'Ambléon,ambleon,AMBLÉON,COM,commune,84,Auvergne-Rhône-Alpes,01,Ain,0104,Belley,200040350,CC Bugey Sud,10,Lyon,1300.0,01300,8404.0,01034,01000,NaN,0.0,HORS UNITE URBAINE,H,113,602,6,19.0,589,330.0,940.0,45.748,5.601,45.750,5.594,6,Rural à habitat dispersé,0.0,communes non pôle,Ambléonais,https://fr.wikipedia.org/wiki/fr:Ambléon,https://villedereve.fr/ville/01006-ambleon


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34935 entries, 0 to 34934
Data columns (total 46 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   code_insee                         34935 non-null  object 
 1   nom_standard                       34935 non-null  object 
 2   nom_sans_pronom                    34935 non-null  object 
 3   nom_a                              34935 non-null  object 
 4   nom_de                             34935 non-null  object 
 5   nom_sans_accent                    34935 non-null  object 
 6   nom_standard_majuscule             34935 non-null  object 
 7   typecom                            34935 non-null  object 
 8   typecom_texte                      34935 non-null  object 
 9   reg_code                           34935 non-null  int64  
 10  reg_nom                            34935 non-null  object 
 11  dep_code                           34935 non-null  obj

In [12]:
communes = df_communes[[
    "code_insee",
    "nom_standard",
    "code_postal",
    "dep_code",   # optional
    "dep_nom"     # optional
]].copy()

# Standardize text for matching
communes["nom_standard"] = communes["nom_standard"].str.upper().str.strip()
communes["code_postal"] = communes["code_postal"].astype(str).str.zfill(5)

In [13]:
communes.head()

,code_insee,nom_standard,code_postal,dep_code,dep_nom
0,01001,L'ABERGEMENT-CLÉMENCIAT,1400.0,01,Ain
1,01002,L'ABERGEMENT-DE-VAREY,1640.0,01,Ain
2,01004,AMBÉRIEU-EN-BUGEY,1500.0,01,Ain
3,01005,AMBÉRIEUX-EN-DOMBES,1330.0,01,Ain
4,01006,AMBLÉON,1300.0,01,Ain


In [17]:
import pandas as pd
# create a dataframe with these columns:
# columns: code_insee,	nom_standard,	nom_sans_pronom,	nom_a,	nom_de,	nom_sans_accent,	nom_standard_majuscule, reg_code,	reg_nom,	dep_code,	dep_nom
df_communes_names = df_communes[[
    "code_insee",	
    "nom_standard",	
    "nom_sans_pronom",	
    "nom_a",
    "nom_de",
    "nom_sans_accent",
    "nom_standard_majuscule",
    "reg_code",
    "reg_nom",
    "dep_code",
    "dep_nom"
    ]].copy()

display(df_communes_names.head())
# save in csv
df_communes_names.to_csv("../data/processed/communes_names.csv", index=False)

,code_insee,nom_standard,nom_sans_pronom,nom_a,nom_de,nom_sans_accent,nom_standard_majuscule,reg_code,reg_nom,dep_code,dep_nom
0,01001,L'Abergement-Clémenciat,Abergement-Clémenciat,à Abergement-Clémenciat,de l'Abergement-Clémenciat,l-abergement-clemenciat,L'ABERGEMENT-CLÉMENCIAT,84,Auvergne-Rhône-Alpes,01,Ain
1,01002,L'Abergement-de-Varey,Abergement-de-Varey,à Abergement-de-Varey,de l'Abergement-de-Varey,l-abergement-de-varey,L'ABERGEMENT-DE-VAREY,84,Auvergne-Rhône-Alpes,01,Ain
2,01004,Ambérieu-en-Bugey,Ambérieu-en-Bugey,à Ambérieu-en-Bugey,d'Ambérieu-en-Bugey,amberieu-en-bugey,AMBÉRIEU-EN-BUGEY,84,Auvergne-Rhône-Alpes,01,Ain
3,01005,Ambérieux-en-Dombes,Ambérieux-en-Dombes,à Ambérieux-en-Dombes,d'Ambérieux-en-Dombes,amberieux-en-dombes,AMBÉRIEUX-EN-DOMBES,84,Auvergne-Rhône-Alpes,01,Ain
4,01006,Ambléon,Ambléon,à Ambléon,d'Ambléon,ambleon,AMBLÉON,84,Auvergne-Rhône-Alpes,01,Ain


<style>
h3 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

### DEAL with INVALID VALUES - Valeurs Foncieres

In [19]:
# ----------------------------------------
# Column: transaction_type
# ----------------------------------------
# fix transaction_type encoding issues to ensure consistent values
# replace values to english
df['transaction_type'] = df['transaction_type'].replace({
        "Vente": "Sale",
        "Vente en l'Ã©tat futur d'achÃ¨vement": "Sale in future state of completion",
        "Vente terrain Ã\xa0 bÃ¢tir": "Sale of unbuilt land",
        "Echange": "Exchange",
        "Adjudication": "Auction",
        "Expropriation": "Expropriation"})


# --------------------------
# Column: street_number
# --------------------------
# nulls in street_number are common and not necessarily invalid, so we will keep them as is for now



# ------------------------
# Column: btq_code
# ------------------------
# fix btq_code inconsistencies: replace invalid values with NaN and standardize 'b' to 'B'
invalid_values = ['-', '.', '*']
df["btq_code"] = df["btq_code"].replace(invalid_values, pd.NA)
df['btq_code'] = df['btq_code'].replace({ 'b': 'B'})


# --------------------------------------------------------------------------------------
# Column: commune_name
# --------------------------------------------------------------------------------------

# --- REPLACE COMMUNE NAMES in DataFrame with OFFICIAL NAMES FROM COMMUNES DATASET TO ENSURE CONSISTENCY ---
# load communes dataset with INSEE codes and official names
commune_names = pd.read_csv("../data/processed/communes_names.csv")

# create mapping for verifying commune_name in dataframe
#commune_mapping = 

# replace commune names in the main dataset using the mapping dictionary
#df["commune_name"] = df["commune_name"].replace(commune_mapping)

# -------------------------------
# Column: building_real_surface
# -------------------------------
# because it is probably not a built property if building_real_surface is NaN
# as building area is not relevant for land sales
# fill NaN with 0 
df["building_real_surface"] = df["building_real_surface"].fillna(0)

# --------------------------
# Column: main_rooms_count
# --------------------------
# because it is probably not a built property if main_rooms_count is NaN
# fill NaN with 0 
df["main_rooms_count"] = df["main_rooms_count"].fillna(0)

# -----------------------
# Column: surface_type
# -----------------------
# surface_type: replace 'bati' with 'built' and 'terrain' with 'land'
df['surface_type'] = df['surface_type'].replace({
    "bati": "built",    
    "terrain": "land"})


# --------------------------------------------------------------------------------------------
# SAVE - InvalidValues ValeursFoncieres
# --------------------------------------------------------------------------------------------
df.to_csv("../data/processed/INTERIM_02_InvalidValues_ValeursFoncieres.csv", index=False)
print(f"✅ Dataset with invalid values handled and saved at ../data/processed/INTERIM_02_InvalidValues_ValeursFoncieres.csv")

✅ Dataset with invalid values handled and saved at ../data/processed/INTERIM_02_InvalidValues_ValeursFoncieres.csv


<hr>

## 3 - DATA TYPES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DISPLAY TYPES - Valeurs Foncieres

In [2]:
# Load the dataset
df = pd.read_csv("../data/processed/INTERIM_02_InvalidValues_ValeursFoncieres.csv")

# Display DataFrame
print("DataFrame:")
print(df.shape[0], "rows and", df.shape[1], "columns")
display(df.head())

C:\Users\sboub\AppData\Local\Temp\ipykernel_13488\2642735328.py:2: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/INTERIM_02_InvalidValues_ValeursFoncieres.csv")


DataFrame:
19908349 rows and 17 columns


,transaction_date,transaction_type,property_value,street_number,btq_code,street_name,postal_code,commune_name,number_of_lots,property_type,building_real_surface,main_rooms_count,land_surface,effective_surface,value_per_m2,surface_type,department_code
0,01/07/2020,Sale,31234.16,NaN,NaN,SAINT JULIEN,1560.0,SAINT-JULIEN-SUR-REYSSOUZE,0,NaN,0.0,0.0,1192.0,1192.0,26.203154,land,15
1,01/07/2020,Sale,278000.00,NaN,NaN,A LA PEROUSE,1250.0,CORVEISSIAT,0,NaN,0.0,0.0,10092.0,10092.0,27.546572,land,12
2,01/07/2020,Sale,278000.00,NaN,NaN,A LA PEROUSE,1250.0,CORVEISSIAT,0,NaN,0.0,0.0,4570.0,4570.0,60.831510,land,12
3,01/07/2020,Sale,278000.00,NaN,NaN,AUX COMMUNS,1250.0,CORVEISSIAT,0,NaN,0.0,0.0,5750.0,5750.0,48.347826,land,12
4,01/07/2020,Sale,278000.00,NaN,NaN,EN COMBARNAUD,1250.0,SIMANDRE-SUR-SURAN,0,NaN,0.0,0.0,648170.0,648170.0,0.428900,land,12


In [3]:
# print data types and unique values count for each column in df
print("Data types check:")
for col in df.columns:
    dtype = df[col].dtype
    unique_vals = df[col].nunique()
    print(f"  ➡️{col:<25} {str(dtype):<10} [{unique_vals}] unique values")

Data types check:
  ➡️transaction_date          object     [1804] unique values
  ➡️transaction_type          object     [6] unique values
  ➡️property_value            float64    [466667] unique values
  ➡️street_number             float64    [9120] unique values
  ➡️btq_code                  object     [36] unique values
  ➡️street_name               object     [1125219] unique values
  ➡️postal_code               float64    [5871] unique values
  ➡️commune_name              object     [31569] unique values
  ➡️number_of_lots            int64      [123] unique values
  ➡️property_type             object     [4] unique values
  ➡️building_real_surface     float64    [9184] unique values
  ➡️main_rooms_count          float64    [75] unique values
  ➡️land_surface              float64    [85686] unique values
  ➡️effective_surface         float64    [85082] unique values
  ➡️value_per_m2              float64    [3967896] unique values
  ➡️surface_type              object     [2] unique 

In [4]:
# Loop through each column and print unique values
print(100 * "-")
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"➡️ {col} ({len(unique_vals)}) 🧾 unique values:")
    print(f"{unique_vals}")
    print(100 * "-")

----------------------------------------------------------------------------------------------------
➡️ transaction_date (1804) 🧾 unique values:
['01/07/2020' '04/07/2020' '02/07/2020' ... '01/05/2025' '18/05/2025'
 '05/01/2025']
----------------------------------------------------------------------------------------------------
➡️ transaction_type (6) 🧾 unique values:
['Sale' 'Sale in future state of completion' 'Sale of unbuilt land'
 'Exchange' 'Auction' 'Expropriation']
----------------------------------------------------------------------------------------------------
➡️ property_value (466667) 🧾 unique values:
[3.123416e+04 2.780000e+05 3.985000e+03 ... 7.088090e+05 5.503861e+05
 2.441758e+07]
----------------------------------------------------------------------------------------------------
➡️ street_number (9121) 🧾 unique values:
[  nan  347. 1084. ... 8562. 8301. 8423.]
----------------------------------------------------------------------------------------------------
➡️ btq

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### CONVERT - Valeurs Foncieres

| Field Name                                 | Python data type             |
|--------------------------------------------|------------------------------|
| **transaction_date**                          | `object` to `date`??           |
| **street_number**                                | `float64` to `object`         |
| **postal_code**                            | `float64` to `object`        |
| **main_rooms**              | `float64` to `int64`         |


In [ ]:
# convert df columns types as follows:
#df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors='coerce')
df["street_number"] = df["street_number"].astype('object')
df["postal_code"] = df["postal_code"].astype('object')
df["main_rooms_count"] = df["main_rooms_count"].astype('int64')


# Display Converted column names
print("Converted column names:")
display(df.info())

# Save dataset in a new CSV file
df.to_csv("../data/processed/INTERIM_03_DataType_ValeursFoncieres.csv", index=False)
print(f"✅ Dataset converted to appropriate data types and saved at ../data/processed/INTERIM_03_DataType_ValeursFoncieres.csv")

Converted column names:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19908349 entries, 0 to 19908348
Data columns (total 17 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   transaction_date       object 
 1   transaction_type       object 
 2   property_value         float64
 3   street_number          float64
 4   btq_code               object 
 5   street_name            object 
 6   postal_code            object 
 7   commune_name           object 
 8   number_of_lots         int64  
 9   property_type          object 
 10  building_real_surface  float64
 11  main_rooms_count       int64  
 12  land_surface           float64
 13  effective_surface      float64
 14  value_per_m2           float64
 15  surface_type           object 
 16  department_code        object 
dtypes: float64(6), int64(2), object(9)
memory usage: 2.5+ GB


None

✅ Dataset converted to appropriate data types and saved at ../data/processed/INTERIM_03_DataType_ValeursFoncieres.csv


<hr>

## 4 - NULL VALUES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DISPLAY NULLs - Valeurs Foncieres

In [1]:
import pandas as pd

# Load dataset with converted data types
df = pd.read_csv("../data/processed/INTERIM_03_DataType_ValeursFoncieres.csv")

# Loop through each column and print the number of nulls in each column
for col in df.columns:
    null_count = df[col].isnull().sum()
    print(f"[{col}]: [{null_count}] null values")

print(100*"-")
# display data types and missing values of each column in df
df.info()

C:\Users\sboub\AppData\Local\Temp\ipykernel_8308\2047978848.py:4: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/INTERIM_03_DataType_ValeursFoncieres.csv")


[transaction_date]: [0] null values
[transaction_type]: [0] null values
[property_value]: [0] null values
[street_number]: [7360821] null values
[btq_code]: [19012414] null values
[street_name]: [150834] null values
[postal_code]: [151147] null values
[commune_name]: [0] null values
[number_of_lots]: [0] null values
[property_type]: [8091437] null values
[building_real_surface]: [0] null values
[main_rooms_count]: [0] null values
[land_surface]: [6383746] null values
[effective_surface]: [3733648] null values
[value_per_m2]: [3734243] null values
[surface_type]: [0] null values
[department_code]: [0] null values
----------------------------------------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19908349 entries, 0 to 19908348
Data columns (total 17 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   transaction_date       object 
 1   transaction_type       object 
 2   property_value 

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DEAL with NULLs - Valeur Foncieres

In [ ]:
# --------------------------------------------------------------------------------
# Column: property_value (euros)
# --------------------------------------------------------------------------------
# drop nulls in sale_price_eur column as it's a critical column for analysis
df = df.dropna(subset=["property_value"])

# --------------------------------------------------------------------------------
# Column: building_real_surface
# --------------------------------------------------------------------------------
# Fill NaN with 0
df["building_real_surface"] = df["building_real_surface"].fillna(0)

# --------------------------------------------------------------------------------
# Column: property_type
# --------------------------------------------------------------------------------
# Fill NaN with "Unknown" to keep all rows for analysis, 
# as property_type is important for analysis and we don't want to drop rows with missing property_type
df["property_type"] = df["property_type"].fillna("Unknown")

# --------------------------------------------------------------------------------
# Column: street_number
# --------------------------------------------------------------------------------
# convert type to object
df["street_number"] = df["street_number"].astype('object')
# Fill NaN with "Unknown" & type as object
df["street_number"] = df["street_number"].fillna("Unknown")


# --------------------------------------------------------------------------------
# Column: main_rooms_count
# --------------------------------------------------------------------------------
# main_rooms is only relevant for houses/apartments
# fill NaN with 0 if property_type is terrain
df["main_rooms_count"] = df.apply(lambda row: 0 if pd.isna(row["main_rooms_count"]) and row["property_type"] == "land" else row["main_rooms_count"], axis=1)


# --------------------------------------------------------------------------------
# Column: property_value (euros)
# --------------------------------------------------------------------------------
# drop nulls in sale_price_eur column as it's a critical column for analysis
df = df.dropna(subset=["property_value"])


# --------------------------------------------------------------------------------
# Column: land_surface
# --------------------------------------------------------------------------------
# Fill NaN with 0 if building_real_surface is not null
df["land_surface"] = df.apply(lambda row: 0 if pd.isna(row["land_surface"]) and not pd.isna(row["building_real_surface"]) else row["land_surface"], axis=1)

# --------------------------------------------------------------------------------
# Column: building_real_surface
# --------------------------------------------------------------------------------
# Fill NaN with 0
df["building_real_surface"] = df["building_real_surface"].fillna(0)

# --------------------------------------------------------------------------------
# Column: property_type
# --------------------------------------------------------------------------------
# Fill NaN with "Unknown" to keep all rows for analysis, 
# as property_type is important for analysis and we don't want to drop rows with missing property_type
df["property_type"] = df["property_type"].fillna("Unknown")

# --------------------------------------------------------------------------------
# Column: street_number
# --------------------------------------------------------------------------------
# convert type to object
df["street_number"] = df["street_number"].astype('object')
# Fill NaN with "Unknown" & type as object
df["street_number"] = df["street_number"].fillna("Unknown")


# --------------------------------------------------------------------------------
# Column: main_rooms_count
# --------------------------------------------------------------------------------
# main_rooms is only relevant for houses/apartments
# fill NaN with 0 if property_type is terrain
df["main_rooms_count"] = df.apply(lambda row: 0 if pd.isna(row["main_rooms_count"]) and row["property_type"] == "land" else row["main_rooms_count"], axis=1)

# --------------------------------------------------------------------------------
# Column: btq_code
# --------------------------------------------------------------------------------
df["btq_code"] = df["btq_code"].fillna("None")

# --------------------------------------------------------------------------------
# Column: street_name
# --------------------------------------------------------------------------------
df["street_name"] = df["street_name"].fillna("Unknown")

# --------------------------------------------------------------------------------
# Column: postal_code
# --------------------------------------------------------------------------------
df["postal_code"] = df["postal_code"].fillna("Unknown")


# --------------------------------------------------------------------------------
# Save dataset in a new CSV file
# --------------------------------------------------------------------------------
df.to_csv("../data/processed/INTERIM_04_Nulls_ValeursFoncieres.csv", index=False)
print(f"✅ Dataset with null values handled and saved at ../data/processed/INTERIM_04_Nulls_ValeursFoncieres.csv")

✅ Dataset with null values handled and saved at ../data/processed/INTERIM_04_Nulls_ValeursFoncieres.csv


In [2]:
import pandas as pd

# Load dataset with nulls handled
df = pd.read_csv("../data/processed/INTERIM_04_Nulls_ValeursFoncieres.csv")

# Loop through each column and print the number of nulls in each column
for col in df.columns:
    null_count = df[col].isnull().sum()
    print(f"[{col}]: [{null_count}] null values")

print(100*"-")
# display data types and missing values of each column in df
df.info()

C:\Users\sboub\AppData\Local\Temp\ipykernel_17304\2309171356.py:4: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/INTERIM_04_Nulls_ValeursFoncieres.csv")


[transaction_date]: [0] null values
[transaction_type]: [0] null values
[property_value]: [0] null values
[street_number]: [0] null values
[btq_code]: [19012414] null values
[street_name]: [150834] null values
[postal_code]: [151147] null values
[commune_name]: [0] null values
[number_of_lots]: [0] null values
[property_type]: [0] null values
[building_real_surface]: [0] null values
[main_rooms_count]: [0] null values
[land_surface]: [6383746] null values
[effective_surface]: [3733648] null values
[value_per_m2]: [3734243] null values
[surface_type]: [0] null values
[department_code]: [0] null values
----------------------------------------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19908349 entries, 0 to 19908348
Data columns (total 17 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   transaction_date       object 
 1   transaction_type       object 
 2   property_value         floa

<hr>

## 5 - DUPLICATES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DISPLAY DUPLICATE ROWS - Valeurs Foncieres

In [3]:
import pandas as pd

# load dataset with nulls handled
df = pd.read_csv("../data/processed/INTERIM_04_Nulls_ValeursFoncieres.csv")

# Check for duplicate rows in the DataFrame
duplicate_rows = df[df.duplicated()]

# Print the number of duplicate rows
print(f"Number of duplicate rows: {duplicate_rows.shape[0]}")

# Display the duplicate rows if any exist
if not duplicate_rows.empty:
    print("\nDuplicate rows:")
    display(duplicate_rows)
else:
    print("\nNo duplicate  rows found.")

C:\Users\sboub\AppData\Local\Temp\ipykernel_17304\158086586.py:4: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/INTERIM_04_Nulls_ValeursFoncieres.csv")


Number of duplicate rows: 2538771

Duplicate rows:


,transaction_date,transaction_type,property_value,street_number,btq_code,street_name,postal_code,commune_name,number_of_lots,property_type,building_real_surface,main_rooms_count,land_surface,effective_surface,value_per_m2,surface_type,department_code
44,02/07/2020,Sale in future state of completion,360000.0,12.0,NaN,DU 23EME R.I.,1000.0,BOURG-EN-BRESSE,1,Dépendance,0.0,0,NaN,NaN,NaN,land,10
60,01/07/2020,Sale,270000.0,6.0,NaN,CREVE COEUR,1000.0,BOURG-EN-BRESSE,1,Dépendance,0.0,0,NaN,NaN,NaN,land,10
91,09/07/2020,Sale in future state of completion,20000.0,street_number,NaN,NaN,NaN,SAINT-DENIS-LES-BOURG,1,Unknown,0.0,0,NaN,NaN,NaN,land,na
95,09/07/2020,Sale in future state of completion,45000.0,street_number,NaN,NaN,NaN,SAINT-DENIS-LES-BOURG,1,Unknown,0.0,0,NaN,NaN,NaN,land,na
96,09/07/2020,Sale in future state of completion,45000.0,street_number,NaN,NaN,NaN,SAINT-DENIS-LES-BOURG,1,Unknown,0.0,0,NaN,NaN,NaN,land,na
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19908303,30/06/2025,Sale,250000.0,8.0,NaN,SAINT SEBASTIEN,75011.0,PARIS 11,1,Appartement,15.0,1,NaN,15.0,16666.666667,built,75
19908312,17/06/2025,Sale,700000.0,3.0,NaN,DE LA NATION,75011.0,PARIS 11,1,Dépendance,0.0,0,NaN,NaN,NaN,land,75
19908326,26/06/2025,Sale,540000.0,88.0,NaN,VICTOR HUGO,75016.0,PARIS 16,2,Dépendance,0.0,0,NaN,NaN,NaN,land,75
19908330,17/06/2025,Sale,592028.0,59.0,NaN,ERLANGER,75016.0,PARIS 16,2,Dépendance,0.0,0,NaN,NaN,NaN,land,75


<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DEAL with DUPLICATE ROWS - Valeurs Foncieres

In [ ]:
# drop duplicate rows in df
#df = df.drop_duplicates()

# Save dataset in a new CSV file
df.to_csv("../data/processed/INTERIM_05_duplicates_ValeursFoncieres.csv", index=False)
print(f"✅ Dataset with duplicates handled and saved at ../data/processed/INTERIM_05_duplicates_ValeursFoncieres.csv")

<hr>

## 🕵️ REVISE & SAVE CLEAN DATA


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

```text
- display head
- display info
- check unique values
- check data types
- check for nulls
- check for duplicates
```

In [ ]:
import pandas as pd

# --------------------------------------------
# Load dataset with duplicates handled
# --------------------------------------------
df = pd.read_csv("../data/processed/INTERIM_05_duplicates_ValeursFoncieres.csv")

# --------------------
# 🔲 DISPLAY - HEAD
# --------------------
# Display df DataFrame
print("Data Revision:")
display(df.head())
print("Dataset Shape:", df.shape[0], "rows and", df.shape[1], "columns\n")
print(100*"-")

# --------------------
# 🔲 DISPLAY - INFO
# --------------------
# display data types and missing values of each column in df
df.info()
print(100*"-")

# -----------------------------
# 🕵️ CHECK - UNIQUE VALUES
# -----------------------------
# Display data types & missing values of each column in df
print("Data Types & Missing Values of Each Column:")
display(df.info())

# Loop through each column and print unique values
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"Column: {col}")
    print(f"Unique values ({len(unique_vals)}): {unique_vals}\n")
print(100*"-")

# -----------------------------
# 🕵️ CHECK - DATA TYPES
# -----------------------------
print(f"\nData types check:")
for col in df.columns:
    dtype = df[col].dtype
    unique_vals = df[col].nunique()
    print(f"  ➡️{col:<25} {str(dtype):<10} [{unique_vals}] unique values")
print(100*"-")

# -----------------------------
# 🕵️ CHECK - NULLS
# -----------------------------
# Loop through each column and print the number of nulls in each column
for col in df.columns:
    null_count = df[col].isnull().sum()
    print(f"[{col}]: [{null_count}] null values")
print(100*"-")


# -----------------------------
# 🕵️ CHECK - DUPLICATES
# -----------------------------
# Check for duplicate rows in the DataFrame
duplicate_rows = df[df.duplicated()]

# Print the number of duplicate rows
print(f"Number of duplicate rows: {duplicate_rows.shape[0]}")
print(100*"-")

# Display the duplicate rows if any exist
if not duplicate_rows.empty:
    print("\nDuplicate rows:")
    display(duplicate_rows)
else:
    print("\nNo duplicate  rows found.")
print(100*"-")

# --------------------------------------------
# Save CLEAN data in a new CSV file
# --------------------------------------------
df.to_csv("../data/processed/CLEAN_ValeursFoncieres_2020-S2_2025-S1.csv", index=False)
print(f"✅ Dataset cleaned and saved at ../data/processed/CLEAN_ValeursFoncieres_2020-S2_2025-S1.csv")